In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import folium

from folium.plugins import HeatMap

In [13]:
import os

os.listdir()

['.ipynb_checkpoints',
 'Class - 21 (Geospatial Analysis).ipynb',
 'geographical working source code.ipynb',
 'Geospatial_Big_Data__sample_rows_.csv',
 'heatmap.html',
 'scatter_map.html']

In [14]:
import pandas as pd

df = pd.read_csv("Geospatial_Big_Data__sample_rows_.csv")

df.head()

,device_id,date,city,lat,lon,elevation_m,category,temperature_C,rainfall_mm,metric_value
0,D0000,2024-01-01,Tokyo,35.705258,139.673853,3.285368,Urban,31.609313,2.734897,51.852692
1,D0000,2024-01-02,Tokyo,35.759214,139.708437,3.285368,Urban,29.588795,0.000000,53.525638
2,D0000,2024-01-03,Tokyo,35.621604,139.616009,3.285368,Urban,31.338142,6.404016,48.829423
3,D0000,2024-01-04,Tokyo,35.582897,139.737659,3.285368,Urban,31.594959,0.000000,48.907782
4,D0000,2024-01-05,Tokyo,35.597423,139.619376,3.285368,Urban,32.134281,0.000000,48.291481


In [15]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (20, 10)

Columns:
['device_id', 'date', 'city', 'lat', 'lon', 'elevation_m', 'category', 'temperature_C', 'rainfall_mm', 'metric_value']


In [16]:
lat_col = "lat"
lon_col = "lon"

print("Latitude:", lat_col)
print("Longitude:", lon_col)

Latitude: lat
Longitude: lon


In [17]:
geo_df = df.dropna(subset=[lat_col, lon_col]).copy()

geo_df[lat_col] = pd.to_numeric(geo_df[lat_col], errors="coerce")
geo_df[lon_col] = pd.to_numeric(geo_df[lon_col], errors="coerce")

geo_df = geo_df.dropna(subset=[lat_col, lon_col])

print("Valid geographic records:", len(geo_df))
geo_df.head()

Valid geographic records: 20


,device_id,date,city,lat,lon,elevation_m,category,temperature_C,rainfall_mm,metric_value
0,D0000,2024-01-01,Tokyo,35.705258,139.673853,3.285368,Urban,31.609313,2.734897,51.852692
1,D0000,2024-01-02,Tokyo,35.759214,139.708437,3.285368,Urban,29.588795,0.000000,53.525638
2,D0000,2024-01-03,Tokyo,35.621604,139.616009,3.285368,Urban,31.338142,6.404016,48.829423
3,D0000,2024-01-04,Tokyo,35.582897,139.737659,3.285368,Urban,31.594959,0.000000,48.907782
4,D0000,2024-01-05,Tokyo,35.597423,139.619376,3.285368,Urban,32.134281,0.000000,48.291481


In [18]:
gdf = gpd.GeoDataFrame(
    geo_df,
    geometry=gpd.points_from_xy(
        geo_df[lon_col],
        geo_df[lat_col]
    ),
    crs="EPSG:4326"
)

gdf.head()

,device_id,date,city,lat,lon,elevation_m,category,temperature_C,rainfall_mm,metric_value,geometry
0,D0000,2024-01-01,Tokyo,35.705258,139.673853,3.285368,Urban,31.609313,2.734897,51.852692,POINT (139.67385 35.70526)
1,D0000,2024-01-02,Tokyo,35.759214,139.708437,3.285368,Urban,29.588795,0.000000,53.525638,POINT (139.70844 35.75921)
2,D0000,2024-01-03,Tokyo,35.621604,139.616009,3.285368,Urban,31.338142,6.404016,48.829423,POINT (139.61601 35.6216)
3,D0000,2024-01-04,Tokyo,35.582897,139.737659,3.285368,Urban,31.594959,0.000000,48.907782,POINT (139.73766 35.5829)
4,D0000,2024-01-05,Tokyo,35.597423,139.619376,3.285368,Urban,32.134281,0.000000,48.291481,POINT (139.61938 35.59742)


In [19]:
print("Number of points:", len(gdf))
print("Coordinate system:", gdf.crs)

gdf[["city", "lat", "lon", "geometry"]]

Number of points: 20
Coordinate system: EPSG:4326


,city,lat,lon,geometry
0,Tokyo,35.705258,139.673853,POINT (139.67385 35.70526)
1,Tokyo,35.759214,139.708437,POINT (139.70844 35.75921)
2,Tokyo,35.621604,139.616009,POINT (139.61601 35.6216)
3,Tokyo,35.582897,139.737659,POINT (139.73766 35.5829)
4,Tokyo,35.597423,139.619376,POINT (139.61938 35.59742)
5,Tokyo,35.593819,139.665519,POINT (139.66552 35.59382)
6,Tokyo,35.604008,139.652513,POINT (139.65251 35.60401)
7,Tokyo,35.627600,139.633565,POINT (139.63356 35.6276)
8,Tokyo,35.714925,139.642936,POINT (139.64294 35.71492)
9,Tokyo,35.609313,139.747637,POINT (139.74764 35.60931)


In [20]:
import folium

center = [
    gdf["lat"].mean(),
    gdf["lon"].mean()
]

m = folium.Map(
    location=center,
    zoom_start=5,
    tiles="CartoDB positron"
)

m

In [21]:
for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        popup=f"{row['city']} | {row['category']}",
        fill=True
    ).add_to(m)

m

In [22]:
from folium.plugins import HeatMap

heat_data = gdf[["lat", "lon"]].values.tolist()

HeatMap(
    heat_data,
    radius=12,
    blur=18,
    max_zoom=12,
    min_opacity=0.2
).add_to(m)

m

In [23]:
m.save("heatmap.html")

In [24]:
print("Heatmap saved successfully.")

Heatmap saved successfully.


In [25]:
scatter_map = folium.Map(
    location=center,
    zoom_start=5,
    tiles="CartoDB positron"
)

for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        popup=f"{row['city']} | {row['category']}",
        fill=True
    ).add_to(scatter_map)

scatter_map

In [26]:
scatter_map.save("scatter_map.html")